# Performance Tips and Optimization

**Learning Objectives:**
- Understand vectorization vs `apply()` vs `iterrows()`
- Use NumPy functions on DataFrames
- Leverage `eval()` and `query()` for fast operations
- Use categorical dtype to save memory
- Chunk large files
- Profile with `%timeit`
- Configure `pd.options.display`
- Know when to use Python lists vs Pandas

## 1. Setup – Large-ish DataFrame

We'll create a dataset with 1 million rows for benchmarking.

In [ ]:
import pandas as pd
import numpy as np

n = 1_000_000
df = pd.DataFrame({
    "A": np.random.randn(n),
    "B": np.random.randint(0, 100, n),
    "C": np.random.choice(["low", "medium", "high"], n),
    "D": np.random.uniform(0, 1000, n)
})
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

## 2. Vectorization vs `apply()` vs `iterrows()`

**Vectorized** operations use fast C loops. `apply()` and `iterrows()` use slow Python loops.

In [ ]:
# Compute: Z = A * D + B

# Vectorized (fast)
%timeit df["Z_vec"] = df["A"] * df["D"] + df["B"]

# apply (slower)
%timeit df["Z_apply"] = df.apply(lambda r: r["A"] * r["D"] + r["B"], axis=1)

# iterrows (slowest – avoid)
def with_iterrows(df):
    vals = []
    for idx, row in df.iterrows():
        vals.append(row["A"] * row["D"] + row["B"])
    return vals

%timeit df["Z_iter"] = with_iterrows(df)

In [ ]:
# Verify results are equal
print("All close?", np.allclose(df["Z_vec"], df["Z_apply"]))
print("Vectorized is fastest, iterrows is slowest.")

## 3. Using NumPy Functions on DataFrames

Directly apply NumPy functions to DataFrame columns (they call the underlying arrays).

In [ ]:
# NumPy operations on columns
df["A_log"] = np.log(np.abs(df["A"]) + 1)
df["B_sqrt"] = np.sqrt(df["B"])
df["D_clip"] = np.clip(df["D"], 100, 900)

print(df[["A", "A_log", "B", "B_sqrt"]].head())

## 4. `eval()` and `query()` for Fast Operations

Use `pd.eval()` / `df.eval()` for expressions and `df.query()` for filters.
They bypass Python overhead by parsing the expression directly.

In [ ]:
# eval for computation (more memory-efficient than vectorized)
%timeit df.eval("Z_eval = A * D + B", inplace=False)

# query for filtering
%timeit df.query("A > 0 and B < 50 and C == 'high'")

# Equivalent boolean indexing
%timeit df[(df["A"] > 0) & (df["B"] < 50) & (df["C"] == "high")]

print("\neval/query can be faster and use less memory.")

## 5. Categorical Data Type – Memory Savings

If a string column has few unique values, convert to `category` dtype.

In [ ]:
# Memory before
mem_before = df.memory_usage(deep=True)["C"]
print(f"Column C memory (object): {mem_before / 1e3:.1f} KB")

# Convert to category
df["C_cat"] = df["C"].astype("category")
mem_after = df.memory_usage(deep=True)["C_cat"]
print(f"Column C memory (category): {mem_after / 1e3:.1f} KB")
print(f"Reduction: {mem_before / mem_after:.1f}x")

In [ ]:
# Convert all eligible columns
df_optimized = df.copy()
for col in df_optimized.select_dtypes(include="object"):
    nunique = df_optimized[col].nunique()
    if nunique / len(df_optimized) < 0.5:  # less than 50% unique
        df_optimized[col] = df_optimized[col].astype("category")
        print(f"Converted {col}: {nunique} categories")

## 6. Chunking Large Files

Process a file in pieces instead of loading it entirely.

In [ ]:
# Write a medium file and read in chunks
df_small = df.head(50000)
df_small.to_csv("_temp_large.csv", index=False)

total = 0
for chunk in pd.read_csv("_temp_large.csv", chunksize=10000):
    total += chunk["A"].sum()
print(f"Sum of A (via chunks): {total:.2f}")

import os
os.remove("_temp_large.csv")

## 7. `inplace` vs Copy

Using `inplace=True` does not always save memory. The modern Pandas idiom uses assignment.

In [ ]:
# Modern idiom: assign back instead of inplace
df["A"] = df["A"].fillna(0)           # instead of df["A"].fillna(0, inplace=True)
df = df.dropna()                       # instead of df.dropna(inplace=True)
print("Use assignment for clarity.")

## 8. `pd.options.display` Settings

Control how DataFrames are displayed.

In [ ]:
# Common display options
print("Default max rows:", pd.options.display.max_rows)
print("Default max columns:", pd.options.display.max_columns)

# Set for this session
pd.set_option("display.max_rows", 20)
pd.set_option("display.precision", 3)
pd.set_option("display.float_format", "{:.3f}".format)

print("\nDataFrame with custom display:")
print(df.head(10))

## 9. Practice Problems

In [ ]:
# Problem 1: Time the difference between vectorized operations
#            and iterrows for: df['X'] = df['A'] * df['B'] + df['D']
#            Report the speed ratio.
import time

df_small = df.head(50000).copy()

start = time.time()
df_small["X_vec"] = df_small["A"] * df_small["B"] + df_small["D"]
vec_time = time.time() - start

start = time.time()
for idx in range(len(df_small)):
    df_small.loc[idx, "X_loop"] = df_small.loc[idx, "A"] * df_small.loc[idx, "B"] + df_small.loc[idx, "D"]
loop_time = time.time() - start

print(f"Vectorized: {vec_time:.4f}s, Loop: {loop_time:.4f}s")
print(f"Vectorized is ~{loop_time / vec_time:.0f}x faster")

In [ ]:
# Problem 2: Reduce memory usage of a DataFrame by converting
#            columns to more efficient dtypes (int8, float32, category).
df_mem = pd.DataFrame({
    "id": np.arange(100000),
    "value": np.random.randn(100000),
    "status": np.random.choice(["active", "inactive", "pending"], 100000),
    "score": np.random.randint(0, 100, 100000)
})

print("Before memory:", df_mem.memory_usage(deep=True).sum() / 1e6, "MB")

df_mem["id"] = df_mem["id"].astype("int32")
df_mem["value"] = df_mem["value"].astype("float32")
df_mem["status"] = df_mem["status"].astype("category")
df_mem["score"] = df_mem["score"].astype("int8")

print("After memory:", df_mem.memory_usage(deep=True).sum() / 1e6, "MB")

## 10. Summary

| Technique | Benefit |
|-----------|---------|
| Vectorization | 10-100x faster than loops |
| `eval()` / `query()` | Memory-efficient expressions |
| Categorical dtype | Huge memory savings for low-cardinality strings |
| Chunking | Process files larger than RAM |
| `pd.options.display` | Control printed output |
| Downcast numeric dtypes | Reduce memory for int/float columns |
| Avoid `iterrows()` | Use vectorized or `apply()` instead |